In [12]:
! python --version

Python 3.12.8


In [13]:
from __future__ import annotations

"""Utility helpers for the recipe chatbot backend.

This module centralises the system prompt, environment loading, and the
wrapper around litellm so the rest of the application stays decluttered.
"""

import os
from typing import Final
import litellm  # type: ignore
from dotenv import load_dotenv

# Ensure the .env file is loaded as early as possible.
load_dotenv(override=False)

# --- Constants -------------------------------------------------------------------

meal_type_options = ["entrée", "dessert", "main", "beverage"]
dietary_preference_options = ["vegan", "gluten-free", "keto", "dairy-free", "vegetarian", "paleo", "diabetic-friendly", "nut-free", "raw vegan", "pescatarian", "whole30", "low-sodium", "low-carb", "sugar-free", "halal", "kosher"]
difficulty_options = ["easy", "medium", "hard", "very_hard"]
mealtime_options = ["breakfast", "lunch", "dinner", "snack"]
time_required_options = ["under_30_minutes", "30_to_60_minutes", "over_1_hour"]
COMBINATION_COUNT: Final[int] = 40

SYSTEM_PROMPT: Final[str] = f'''\
Given the following key Japanese recipe dimensions:
- Difficulty: one of {difficulty_options}
- Dietary Preference: one of {dietary_preference_options}
- Meal Type: one of {meal_type_options}
- Mealtime: one of {mealtime_options}
- Time Required: one of {time_required_options}

Can you provide a list of exactly {COMBINATION_COUNT} combinations of these dimensions? They have to make sense together.

For each combination, also provide a brief persona description that would fit that combination.

For example, here are a few combinations based on a particular persona:

1) Persona: "Beginner cook looking for quick and easy meals"
   - Difficulty: easy
   - Dietary Preference: paleo
   - Meal Type: main
   - Mealtime: dinner
   - Time Required: under_30_minutes

2) Persona: "Health-conscious individual seeking vegetarian options"
   - Difficulty: medium
   - Dietary Preference: vegetarian
   - Meal Type: entrée
   - Mealtime: lunch
   - Time Required: 30_to_60_minutes

3) Persona: "Gourmet chef interested in complex recipes"
    - Difficulty: very_hard
    - Dietary Preference: raw vegan
    - Meal Type: main
    - Mealtime: dinner
    - Time Required: over_1_hour

Remember, that we only need {COMBINATION_COUNT} combinations.
'''

# Fetch configuration *after* we loaded the .env file.
MODEL_NAME: Final[str] = os.environ.get("MODEL_NAME", "gpt-4o-mini")

def get_dimension_combinations() -> list[tuple[str, tuple[str, str, str, str, str]]]:
    """
    Use the SYSTEM_PROMPT and an LLM call to return a realistic list of {COMBINATION_COUNT} unique combinations of personas, and key Japanese
    recipe dimensions (difficulty, dietary_preference, meal_type, mealtime, time_required) as tuples.
    
    Returns:
        List of tuples where each tuple is (persona_description, (difficulty, dietary_preference, meal_type, mealtime, time_required))
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Please provide the list as a Python list of tuples, where each tuple contains:\n(persona_description, (difficulty, dietary_preference, meal_type, mealtime, time_required))\n\nFor example:\n[('Beginner cook looking for quick meals', ('easy', 'omnivore', 'main', 'dinner', 'under_30_minutes')), ...]"}
    ]
    completion = litellm.completion(
        model=MODEL_NAME,
        messages=messages,
    )
    import ast
    import re
    # Extract the list of tuples from the assistant's reply
    reply = completion["choices"][0]["message"]["content"].strip()
    # Try to extract the first Python list of tuples from the reply
    match = re.search(r'\[.*\]', reply, re.DOTALL)
    if match:
        list_str = match.group(0)
        try:
            result = ast.literal_eval(list_str)
            if isinstance(result, list) and all(isinstance(t, tuple) and len(t) == 2 for t in result):
                # Validate structure: (persona_string, (5-tuple of dimensions))
                valid = all(
                    isinstance(t[0], str) and 
                    isinstance(t[1], tuple) and 
                    len(t[1]) == 5 
                    for t in result
                )
                if valid:
                    return result
        except Exception as e:
            print(f"Error parsing result: {e}")
            pass
    # Fallback: return the raw reply if parsing fails
    return reply

In [15]:
dimension_combinations = get_dimension_combinations()
length = len(dimension_combinations) if isinstance(dimension_combinations, list) else 'N/A'
print(f"Number of combinations: {length}")

Number of combinations: 39


In [16]:
dimension_combinations

[('Beginner cook looking for quick paleo dinners',
  ('easy', 'paleo', 'main', 'dinner', 'under_30_minutes')),
 ('Busy professional wanting gluten-free breakfast',
  ('easy', 'gluten-free', 'entrée', 'breakfast', 'under_30_minutes')),
 ('Diabetic-friendly snack seeker with medium cooking skills',
  ('medium', 'diabetic-friendly', 'snack', 'snack', '30_to_60_minutes')),
 ('Low-carb ketogenic dinner planner who likes challenges',
  ('medium', 'keto', 'main', 'dinner', '30_to_60_minutes')),
 ('Advanced cook exploring vegan desserts',
  ('hard', 'vegan', 'dessert', 'snack', '30_to_60_minutes')),
 ('Health-conscious pescatarian needing easy lunch ideas',
  ('easy', 'pescatarian', 'main', 'lunch', 'under_30_minutes')),
 ('Vegetarian looking for medium-difficulty dinner recipes',
  ('medium', 'vegetarian', 'main', 'dinner', '30_to_60_minutes')),
 ('Keto enthusiast preparing very hard dinner meals',
  ('very_hard', 'keto', 'main', 'dinner', 'over_1_hour')),
 ('Raw vegan interested in challengi

In [17]:
import asyncio

async def generate_single_query_async(persona: str, dimensions: tuple[str, str, str, str, str]) -> str:
    """Generate a single natural language query for one persona-dimension combination asynchronously."""
    difficulty, dietary_preference, meal_type, mealtime, time_required = dimensions
    
    prompt = f'''You are helping generate a realistic user query for a recipe chatbot.

Given this user persona: "{persona}"
And these recipe requirements:
- Difficulty level: {difficulty}
- Dietary preference: {dietary_preference}
- Meal type: {meal_type}
- Mealtime: {mealtime}
- Time required: {time_required}

Write a single, natural user query that someone with this persona might ask a recipe chatbot. The query should reflect their cooking level, dietary needs, and time constraints,
but should sound natural and conversational (not mentioning the specific dimension names). Natural language have a lot of imperfections, so feel free to include small typos or
colloquial phrases. Mis-spellings and informal language are welcome. Mixed capitalisations are also common in natural language queries. You don't have to allways start the
question with `Hey`. Also, some queries are quite terse and to the point, while others may be more elaborate.

Return only the query text, no additional formatting or explanation.'''
    
    messages = [
        {"role": "system", "content": "You are an expert at generating realistic, conversational user queries for a recipe chatbot."},
        {"role": "user", "content": prompt}
    ]
    
    try:
        completion = await asyncio.get_event_loop().run_in_executor(
            None, 
            lambda: litellm.completion(model=MODEL_NAME, messages=messages)
        )
        return completion["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"Error generating query for {persona[:30]}...: {e}")
        return f"Error generating query for {persona}"

async def generate_natural_language_queries_async(combinations: list[tuple[str, tuple[str, str, str, str, str]]], n: int = 1, max_combinations: int = None) -> list[str]:
    """
    Use async to generate realistic natural language user queries for each combination in parallel.
    
    Args:
        combinations: List of (persona, dimensions) tuples
        n: Number of queries to generate per combination
        max_combinations: Maximum number of combinations to use (None = use all)
    
    Returns:
        List of generated queries
    """
    # Select combinations to use
    if max_combinations:
        selected_combinations = combinations[:max_combinations]
    else:
        selected_combinations = combinations
    
    total_queries = len(selected_combinations) * n
    print(f"Generating {n} queries for each of {len(selected_combinations)} combinations ({total_queries} total queries)...")
    
    tasks = []
    for persona, dimensions in selected_combinations:
        # Generate n queries for each combination
        for i in range(n):
            tasks.append(generate_single_query_async(persona, dimensions))
    
    queries = await asyncio.gather(*tasks)
    return queries

In [18]:
# In Jupyter, you can await async functions directly in cells
# Generate multiple queries per combination
# Parameters: n=number of queries per combination, max_combinations=limit combinations used
queries = await generate_natural_language_queries_async(dimension_combinations, n=2)
print(f"\nGenerated {len(queries)} user queries:")
print("="*50)
for i, q in enumerate(queries, 1):
    print(f"{i}. {q}")
    print()

Generating 2 queries for each of 39 combinations (78 total queries)...

Generated 78 user queries:
1. Can u suggest some easy paleo dinner recipes I can whip up in less than 30 mins? I’m new to cooking so nothing too complicated please!

2. I’m just starting out with cooking, any quick and easy paleo dinner ideas I can make in like 20-30 mins?

3. Can u recomend some easy gluten free breakfast ideas I can make real quick? don’t have more than half an hour in the morning!

4. Can u suggest some easy gluten free breakfast ideas that I can whip up in like 20 minutes? I’m kinda new to cooking and super busy mornings!

5. Can you suggest some diabetic-friendly snack recipes that aren’t too basic but also not super hard? I’ve got about 30 to 60 mins to cook. Thanks!

6. I’m lookin for a diabetic-friendly snack recipe that’s not too basic, maybe something I can whip up in about 45 mins. Got any ideas? I’m pretty comfy in the kitchen but nothing too fancy.

7. Hey, got any keto-friendly main d

In [19]:
# Export queries to CSV file
import csv
import os

# Create the data directory if it doesn't exist
os.makedirs('../../data', exist_ok=True)

# Write queries to CSV with the specified schema
csv_path = '../../data/sample_queries_hw3.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header
    writer.writerow(['id', 'query'])
    
    # Write queries with sequential IDs
    for i, query in enumerate(queries, 1):
        writer.writerow([i, query])

print(f"Exported {len(queries)} queries to {csv_path}")

Exported 78 queries to ../../data/sample_queries_hw3.csv


### Generating sample queries using Bulk Test Tool

Now that the sample_queries_hw3.csv has been created, run the bulk test script to generate results for homework 3:
```bash
python scripts/bulk_test_hw3.py
```

This will generate a new CSV file with the results of the bulk test for homework 3 in the `results/` directory. We can then run the data viewer annotation tool to review and annotate the results, for the Open Coding part of the analysis.

### Running the Data Viewer for Open Coding Analysis
See instructions on running Data Viewer in this [Readme file](./data_viewer/README.md).

1) Load the results CSV file generated by `bulk_test_hw3.py`
2) For each generated recipe, write your Open codes in the text box provided
3) Save the open codes CSV upon finishing the Open Code Analysis


In [ ]:
import pandas as pd
import re
from typing import List

def extract_open_codes_from_csv(csv_path: str) -> List[str]:
    """
    Extract all open codes from the CSV file.
    Handles both single-line codes and bullet list formats.
    """
    df = pd.read_csv(csv_path)
    all_open_codes = []
    
    for _, row in df.iterrows():
        open_codes_text = str(row['open_codes'])
        
        # Skip empty or NaN entries
        if pd.isna(row['open_codes']) or open_codes_text.strip() == '' or open_codes_text == 'nan':
            continue
            
        # Split by common delimiters and clean up
        lines = open_codes_text.split('\n')
        
        for line in lines:
            line = line.strip()
            
            # Skip empty lines
            if not line:
                continue
                
            # Remove bullet points and dashes
            line = re.sub(r'^[-•*]\s*', '', line)
            line = re.sub(r'^[0-9]+\.\s*', '', line)  # Remove numbered lists
            
            # Clean up and add if not empty
            line = line.strip()
            if line and len(line) > 3:  # Ignore very short entries
                all_open_codes.append(line)
    
    return all_open_codes

# Extract open codes from the CSV
csv_path = '/Users/josereyes/Dev/recipe-chatbot/results/opencodes/opencodes_results_20251025_154225.csv'
open_codes = extract_open_codes_from_csv(csv_path)

print(f"Extracted {len(open_codes)} open codes:")
print("="*50)
for i, code in enumerate(open_codes[:10], 1):  # Show first 10
    print(f"{i}. {code}")
print(f"\n... and {len(open_codes) - 10} more open codes.")

In [ ]:
async def perform_axial_coding(open_codes: List[str]) -> str:
    """
    Use LLM to perform axial coding - identify failure modes and categories from open codes.
    This implements a qualitative research strategy to find patterns and themes.
    """
    # Prepare the open codes text for analysis
    codes_text = "\n".join([f"- {code}" for code in open_codes])
    
    prompt = f'''You are a qualitative research expert performing axial coding analysis on open codes from a recipe chatbot evaluation study.

Below are {len(open_codes)} open codes extracted from user evaluations of a Japanese recipe chatbot's responses. These codes represent observations, issues, patterns, and problems identified during the evaluation process.

OPEN CODES:
{codes_text}

TASK: Perform axial coding to identify:

1. **FAILURE MODE CATEGORIES**: Group related codes into broader failure categories (e.g., "Format Issues", "Content Accuracy Problems", "User Experience Issues", etc.)

2. **PATTERN ANALYSIS**: Identify recurring themes and patterns across the codes

3. **PRIORITY RANKING**: Rank the failure modes by frequency and severity

4. **ROOT CAUSE ANALYSIS**: Suggest potential underlying causes for the most critical failure modes

Please provide your analysis in the following format:

## AXIAL CODING ANALYSIS

### Primary Failure Mode Categories
[List the main categories with examples]

### Pattern Analysis
[Describe recurring themes and patterns]

### Priority Ranking
[Rank failure modes by frequency/severity]

### Root Cause Analysis
[Identify potential underlying causes]

### Recommendations
[Suggest improvements based on the analysis]

Focus on actionable insights that could help improve the recipe chatbot system.'''
    
    messages = [
        {"role": "system", "content": "You are an expert qualitative researcher specializing in axial coding and thematic analysis of user experience data."},
        {"role": "user", "content": prompt}
    ]
    
    try:
        completion = await asyncio.get_event_loop().run_in_executor(
            None, 
            lambda: litellm.completion(model=MODEL_NAME, messages=messages)
        )
        return completion["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"Error performing axial coding: {e}")
        return f"Error performing axial coding: {e}"

# Perform axial coding analysis
print("Performing axial coding analysis...")
axial_analysis = await perform_axial_coding(open_codes)
print("\n" + "="*70)
print("AXIAL CODING ANALYSIS RESULTS")
print("="*70)
print(axial_analysis)

In [ ]:
# Save the axial coding analysis to a file
import datetime

# Create analysis directory if it doesn't exist
analysis_dir = '/Users/josereyes/Dev/recipe-chatbot/homeworks/hw2/analysis'
os.makedirs(analysis_dir, exist_ok=True)

# Generate timestamp for filename
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
analysis_filename = f"axial_coding_analysis_{timestamp}.md"
analysis_path = os.path.join(analysis_dir, analysis_filename)

# Save analysis to markdown file
with open(analysis_path, 'w', encoding='utf-8') as f:
    f.write(f"# Axial Coding Analysis\n\n")
    f.write(f"**Generated:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"**Source:** {csv_path}\n")
    f.write(f"**Open Codes Analyzed:** {len(open_codes)}\n\n")
    f.write("---\n\n")
    f.write(axial_analysis)

print(f"Analysis saved to: {analysis_path}")

# Also save the extracted open codes for reference
codes_filename = f"extracted_open_codes_{timestamp}.txt"
codes_path = os.path.join(analysis_dir, codes_filename)

with open(codes_path, 'w', encoding='utf-8') as f:
    f.write(f"# Extracted Open Codes\n\n")
    f.write(f"Total codes: {len(open_codes)}\n")
    f.write(f"Source: {csv_path}\n\n")
    for i, code in enumerate(open_codes, 1):
        f.write(f"{i}. {code}\n")

print(f"Open codes saved to: {codes_path}")